In [ ]:
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.3 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

SystemError: 2c81.so.�() method: bad call flags

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!ls /content

sample_data  spd_finetuned_output  spd_gguf_model  spd_merged_model


In [ ]:
!nvidia-smi

Fri Jun 19 14:55:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════╗
║     Fine-Tuning Qwen2.5-7B-Instruct — SPD & Child Health       ║
║     QLoRA + SFTTrainer | جاهز للتشغيل على Google Colab         ║
╚══════════════════════════════════════════════════════════════════╝

طريقة الاستخدام على Colab:
  1. Runtime → Change runtime type → GPU (T4 / L4 / A100)
  2. ارفع ملف SPD_finetune_dataset.json على /content/
  3. شغّل:  !python spd_finetune_colab.py

أو بالخيارات:
  !python spd_finetune_colab.py --epochs 5 --hf_token "hf_xxx"
  !python spd_finetune_colab.py --merge_only
  !python spd_finetune_colab.py --merge_only --convert_gguf
"""

# ══════════════════════════════════════════════════════════════════
# 0. IMPORTS & ARGUMENT PARSER
# ══════════════════════════════════════════════════════════════════
import os, sys, json, subprocess, shutil, argparse
from datetime import datetime

# ── شبكة أمان ضد BFloat16: نقفل الـ Mixed Precision خالص من أول استيراد ──
# جربنا قبل كده نجبر mixed_precision=fp16 بس البيئة (إعدادات accelerate
# محفوظة على القرص، أو نسخة مكتبة معينة) فضلت بتسرّب bfloat16 جوه
# GradScaler وبيطلع: "_amp_foreach_non_finite_check_and_unscale_cuda
# not implemented for 'BFloat16'". الحل الأضمن: نقفل الـ AMP بالكامل
# (mixed_precision="no") بدل ما نحاول نحدد نوعه. الموديل الأساسي لسه
# بيتحسب بـ fp16 جوه bitsandbytes (compute_dtype) بغض النظر عن ده.
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

# ── اسم السكريبت بأمان ──
# __file__ مش متاح لما الكود ده يتشغل جوه خلية Jupyter/Colab (بدل ما
# يتشغل كملف .py عادي من الترمينال)، وده كان بيسبب NameError في رسائل
# الإرشادات بتاعتنا. بنستخدم قيمة افتراضية آمنة لو __file__ مش موجودة.
SCRIPT_NAME = globals().get("__file__", "spd_finetune_colab.py")


def parse_args():
    p = argparse.ArgumentParser(
        description="Fine-tune Qwen2.5-7B-Instruct for SPD & Child Health"
    )
    p.add_argument("--dataset",     default="/content/SPD_finetune_dataset.json")
    p.add_argument("--output_dir",  default="/content/spd_finetuned_output")
    p.add_argument("--merged_dir",  default="/content/spd_merged_model")
    p.add_argument("--gguf_dir",    default="/content/spd_gguf_model")
    p.add_argument("--base_model",  default="Qwen/Qwen2.5-7B-Instruct")
    p.add_argument("--hf_token",    default="",  help="HuggingFace API token")
    p.add_argument("--epochs",      type=int,   default=3)
    p.add_argument("--lr",          type=float, default=2e-4)
    p.add_argument("--lora_r",      type=int,   default=None,
                   help="LoRA rank (default: auto based on GPU)")
    p.add_argument("--lora_alpha",  type=int,   default=None,
                   help="LoRA alpha (default: 2x lora_r)")
    p.add_argument("--lora_dropout",type=float, default=0.05)
    p.add_argument("--gguf_quant",  default="Q4_K_M",
                   choices=["Q4_0", "Q4_K_M", "Q5_K_M", "Q8_0", "F16"])
    p.add_argument("--skip_train",    action="store_true")
    p.add_argument("--merge_only",    action="store_true")
    p.add_argument("--convert_gguf",  action="store_true")
    p.add_argument("--upload_hf",     action="store_true",
                   help="Upload adapter to HuggingFace Hub after training")
    p.add_argument("--hf_repo",    default="",
                   help="HF repo name, e.g. username/spd-qwen2.5-7b")
    args, _ = p.parse_known_args()
    return args


# ══════════════════════════════════════════════════════════════════
# 1. INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════════
def install_dependencies():
    sep()
    print("  📦 الخطوة 1: تثبيت المكتبات")
    sep()

    pkgs = [
        "transformers>=4.46.0",
        "peft>=0.13.0",
        "trl>=0.12.0",
        "bitsandbytes>=0.44.0",
        "datasets>=3.0.0",
        "accelerate>=1.0.0",
        "sentencepiece",
        "protobuf",
        "scipy",
        "einops",
        "huggingface_hub",
        # نسخة قديمة من torchao (شائعة كمثبّتة مسبقاً على Colab) بتكسر
        # خطوة merge_adapter بخطأ:
        # "Found an incompatible version of torchao... only versions
        #  above 0.16.0 are supported". بنحدّثها هنا عشان الـ merge
        # ينجح من المرة الأولى من غير ما يحتاج fallback.
        "torchao>=0.16.0",
    ]

    def run(cmd, label=""):
        # ── نطبع الـ output مباشرةً بدل ما نخبيه ──
        # capture_output=True كانت بتسبب المشكلة الأصلية: المستخدم
        # بيشوف الـ terminal صامت لدقائق ويظن السكريبت فريز فيضغط
        # Stop. دلوقتي الـ output بيطلع مباشرةً وهو شايف التقدم.
        if label:
            print(f"  ⏳ {label} (ممكن ياخد دقيقتين-5، استنى...)")
        result = subprocess.run(cmd)
        if result.returncode != 0:
            raise RuntimeError(f"pip install failed: {' '.join(cmd)}")
        return result

    try:
        import torch as _torch_check
        print(f"  ✅ PyTorch موجود بالفعل: {_torch_check.__version__}")
        if not _torch_check.cuda.is_available():
            print("  ⚠️  CUDA مش متاح مع النسخة الحالية — هيتم تثبيت نسخة متوافقة")
            raise ImportError
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", "-U",
             "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu121"],
            label="تثبيت PyTorch (أكبر حزمة — ممكن ياخد 3-5 دقايق)")

    run([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs,
        label="تثبيت مكتبات التدريب (transformers, peft, trl, ...)")

    accel_cfg = os.path.expanduser(
        "~/.cache/huggingface/accelerate/default_config.yaml"
    )
    if os.path.exists(accel_cfg):
        os.remove(accel_cfg)
        print("  🧹 تم حذف إعدادات accelerate الافتراضية (كانت ممكن تجبر bf16)")

    print("  ✅ كل المكتبات اتثبتت!")


# ══════════════════════════════════════════════════════════════════
# 2. GPU CHECK & AUTO CONFIG
# ══════════════════════════════════════════════════════════════════
def check_gpu_and_set_config():
    import torch
    sep()
    print("  🖥️  الخطوة 2: فحص GPU وضبط الإعدادات")
    sep()

    if not torch.cuda.is_available():
        raise RuntimeError(
            "❌ مفيش GPU!\n"
            "   روح: Runtime → Change runtime type → GPU"
        )

    gpu        = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_vram  = torch.cuda.mem_get_info(0)[0] / 1024**3
    cc_major, cc_minor = torch.cuda.get_device_capability(0)

    print(f"  🖥️  GPU      : {gpu}")
    print(f"  💾 VRAM كلي : {total_vram:.1f} GB")
    print(f"  ✅ VRAM حر  : {free_vram:.1f} GB")
    print(f"  🔧 Compute Capability: {cc_major}.{cc_minor}")

    is_ampere_or_newer = cc_major >= 8

    # إعدادات تلقائية حسب الـ VRAM الحر الفعلي (مش الكلي) — على Colab
    # المجاني خصوصاً، الـ VRAM "الحر" بيكون أقل من الكلي بسبب عمليات
    # تانية شغالة. بنستخدم free_vram عشان نقفل أي احتمال OOM من الأول.
    if free_vram >= 35:
        cfg = dict(batch=2, grad_accum=8,  max_seq=2048, lora_r=64, tag="A100/H100 🚀")
    elif free_vram >= 20:
        cfg = dict(batch=2, grad_accum=8,  max_seq=1536, lora_r=32, tag="L4/A10G ⚡")
    elif free_vram >= 13:
        cfg = dict(batch=1, grad_accum=16, max_seq=512,  lora_r=16, tag="T4 💡")
    else:
        # GPU ضيقة جداً (أقل من 13GB حر فعلاً، شائع على Colab المجاني
        # لما فيه عمليات تانية شغالة أو الكروت أقل من T4). نقلل كل حاجة
        # للحد الأدنى عشان نتفادى OOM حتى لو على حساب جودة/سرعة التدريب.
        cfg = dict(batch=1, grad_accum=32, max_seq=256,  lora_r=8, tag="VRAM محدودة جداً 🐢")

    cfg["use_tf32"] = is_ampere_or_newer
    cfg["use_bf16"] = False
    cfg["use_fp16"] = False

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
    torch.cuda.empty_cache()

    print(f"\n  📋 وضع التشغيل: {cfg['tag']}")
    print(f"     Batch size      : {cfg['batch']}")
    print(f"     Grad accum      : {cfg['grad_accum']}")
    print(f"     Effective batch : {cfg['batch'] * cfg['grad_accum']}")
    print(f"     Max seq length  : {cfg['max_seq']}")
    print(f"     LoRA rank       : {cfg['lora_r']}")
    print(f"     Precision       : fp16 compute (no AMP/GradScaler — يتجنب خطأ BFloat16)")

    return cfg


# ══════════════════════════════════════════════════════════════════
# 3. LOAD & VALIDATE DATASET
# ══════════════════════════════════════════════════════════════════
def load_dataset(path):
    sep()
    print(f"  📂 الخطوة 3: تحميل الـ Dataset")
    print(f"     المسار: {path}")
    sep()

    if not os.path.exists(path):
        print(f"  ⚠️  الملف مش موجود: {path}")

        try:
            from google.colab import files as colab_files
            print("  📤 ارفع ملف SPD_finetune_dataset.json دلوقتي:")
            uploaded = colab_files.upload()

            if not uploaded:
                raise FileNotFoundError(
                    f"❌ محدش رفع أي ملف.\n"
                    f"   ارفع ملف SPD_finetune_dataset.json على /content/ وأعد المحاولة."
                )

            uploaded_name = next(iter(uploaded.keys()))
            uploaded_path = os.path.join("/content", uploaded_name)

            if uploaded_path != path:
                shutil.copy(uploaded_path, path)
                print(f"  ✅ تم نسخ '{uploaded_name}' إلى: {path}")
            else:
                print(f"  ✅ تم رفع الملف: {path}")

        except ImportError:
            raise FileNotFoundError(
                f"❌ الملف مش موجود: {path}\n"
                f"   ارفع ملف SPD_finetune_dataset.json على /content/ "
                f"أو مرّر المسار الصحيح بـ --dataset"
            )

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not data:
        raise ValueError(f"❌ الـ Dataset فاضي تماماً: {path}")

    valid, errors = 0, []
    for i, item in enumerate(data):
        if "conversations" not in item:
            errors.append(f"Entry {i}: مفيش 'conversations'")
            continue
        roles = {c["role"] for c in item["conversations"]}
        if {"system", "user", "assistant"}.issubset(roles):
            valid += 1
        else:
            missing = {"system", "user", "assistant"} - roles
            errors.append(f"Entry {i}: ناقص roles → {missing}")

    print(f"  ✅ إجمالي الأمثلة : {len(data)}")
    print(f"  ✅ أمثلة صالحة   : {valid}/{len(data)}")
    if errors:
        print(f"  ⚠️  أخطاء ({len(errors)}):")
        for e in errors[:3]:
            print(f"     {e}")
        if len(errors) > 3:
            print(f"     ...و {len(errors)-3} أخطاء تانية")

    if valid == 0:
        raise ValueError(
            "❌ مفيش ولا مثال واحد صالح في الـ Dataset (لازم كل مثال "
            "يحتوي على roles: system, user, assistant). راجع الملف."
        )

    # معاينة أول مثال صالح (مش بالضرورة أول عنصر في الملف، ممكن يكون فاضي/تالف)
    print("\n  📝 مثال من الـ Dataset:")
    for item in data:
        if "conversations" in item and item["conversations"]:
            for conv in item["conversations"]:
                snippet = conv["content"][:100].replace("\n", " ")
                print(f"     [{conv['role'].upper():9s}]: {snippet}...")
            break

    return data


# ══════════════════════════════════════════════════════════════════
# 4. LOAD TOKENIZER
# ══════════════════════════════════════════════════════════════════
def load_tokenizer(base_model, hf_token):
    from transformers import AutoTokenizer
    sep()
    print(f"  🔤 الخطوة 4: تحميل Tokenizer")
    print(f"     النموذج: {base_model}")
    sep()

    tokenizer = AutoTokenizer.from_pretrained(
        base_model,
        trust_remote_code=True,
        padding_side="right",
        token=hf_token or None,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print(f"  ✅ Tokenizer جاهز — Vocab size: {tokenizer.vocab_size:,}")
    return tokenizer


# ══════════════════════════════════════════════════════════════════
# 5. FORMAT DATASET
# ══════════════════════════════════════════════════════════════════
def format_dataset(raw_data, tokenizer, max_seq_len):
    from datasets import Dataset
    sep()
    print("  📊 الخطوة 5: تنسيق الـ Dataset")
    sep()

    formatted, skipped, longest_tokens = [], 0, 0
    for i, item in enumerate(raw_data):
        if "conversations" not in item or not item["conversations"]:
            skipped += 1
            continue
        messages = [{"role": c["role"], "content": c["content"]}
                    for c in item["conversations"]]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            n_tokens = len(tokenizer.encode(text))
            longest_tokens = max(longest_tokens, n_tokens)
            formatted.append({"text": text})
        except Exception as e:
            skipped += 1
            if i < 3:
                print(f"  ⚠️  Entry {i}: {e}")

    if not formatted:
        raise ValueError(
            "❌ بعد التنسيق مفيش ولا مثال صالح. راجع شكل الـ Dataset "
            "(لازم يكون فيه roles: system/user/assistant سليمة)."
        )

    print(f"  ✅ أمثلة منسّقة      : {len(formatted)}")
    print(f"  ⚠️  تم تخطي (خطأ فقط) : {skipped}")
    print(f"  📏 أطول مثال (tokens) : {longest_tokens}")
    if longest_tokens > max_seq_len:
        print(f"  ℹ️  فيه أمثلة أطول من max_seq_length={max_seq_len} — "
              f"هيتم قصّها (truncate) تلقائياً وقت التدريب، مش حذفها")

    dataset = Dataset.from_list(formatted)

    # ── لازم يكون فيه أمثلة كفاية للـ split ──
    # test_size=0.05 بيتفاضل لصفر مثال للـ eval لو الـ dataset صغير
    # (أقل من ~20 مثال)، وده كان بيكسر eval_dataset أو يديله صحة
    # مضللة. بنضمن إن فيه على الأقل مثال واحد للـ eval.
    n = len(dataset)
    test_size = 0.05 if n >= 40 else max(1 / n, 0.05)
    split = dataset.train_test_split(test_size=test_size, seed=42)

    print(f"  📊 Train : {len(split['train'])} | Eval : {len(split['test'])}")
    return split["train"], split["test"]


# ══════════════════════════════════════════════════════════════════
# 6. LOAD MODEL + LORA
# ══════════════════════════════════════════════════════════════════
def load_model_with_lora(base_model, hf_token, lora_r, lora_alpha,
                          lora_dropout, use_bf16=False):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    sep()
    print("  🤖 الخطوة 6: تحميل النموذج بـ QLoRA")
    print("     ⏳ ده بياخد 5-10 دقايق...")
    sep()

    compute_dtype = torch.float16
    print(f"  🔧 Compute dtype: {compute_dtype}")

    torch.set_default_dtype(torch.float32)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("  ⚡ Flash Attention 2 مفعّل")
    except ImportError:
        attn_impl = "eager"
        print("  📌 Eager Attention")

    model = None
    last_load_error = None
    for candidate_impl in dict.fromkeys([attn_impl, "eager"]):
        try:
            model = AutoModelForCausalLM.from_pretrained(
                base_model,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=compute_dtype,
                attn_implementation=candidate_impl,
                token=hf_token or None,
                low_cpu_mem_usage=True,
            )
            if candidate_impl != attn_impl:
                print(f"  ⚠️  فشل تحميل الموديل بـ {attn_impl} — "
                      f"اتحمّل بنجاح بـ {candidate_impl} بدالها")
            break
        except Exception as e:
            last_load_error = e
            print(f"  ⚠️  فشل تحميل الموديل بـ {candidate_impl}: {e}")
            continue
    if model is None:
        raise RuntimeError(
            f"❌ مستحيل تحميل الموديل بأي attention implementation: {last_load_error}"
        )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    converted = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
            converted += 1
    if converted:
        print(f"  🔧 تم تحويل {converted} بارامتر من bfloat16 إلى float16 "
              f"(منع خطأ GradScaler)")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    used_vram = torch.cuda.memory_allocated(0) / 1024**3

    print(f"  ✅ النموذج جاهز!")
    print(f"     Trainable params : {trainable:,} ({100*trainable/total:.2f}%)")
    print(f"     Total params     : {total:,}")
    print(f"     VRAM مستخدم      : {used_vram:.1f} GB")
    return model


# ══════════════════════════════════════════════════════════════════
# 7. TRAIN
# ══════════════════════════════════════════════════════════════════
def train(args, cfg, model, tokenizer, train_dataset, eval_dataset, raw_data):
    from trl import SFTTrainer, SFTConfig
    import torch
    sep()
    print("  🚀 الخطوة 7: التدريب")
    sep()

    print(f"  📋 ملخص الإعدادات:")
    print(f"     Epochs          : {args.epochs}")
    print(f"     Batch size      : {cfg['batch']}")
    print(f"     Grad accum      : {cfg['grad_accum']}")
    print(f"     Effective batch : {cfg['batch'] * cfg['grad_accum']}")
    print(f"     Learning rate   : {args.lr}")
    print(f"     Max seq length  : {cfg['max_seq']}")
    print(f"     LoRA r / alpha  : {args.lora_r} / {args.lora_alpha}")

    import inspect
    sft_params = set(inspect.signature(SFTConfig.__init__).parameters)

    # ── مستويات تقليل تلقائي عند OOM ──
    # لو الـ GPU ضيقة فعلاً (مش بس ضبط مبدئي خاطئ)، ممكن نقع في
    # CUDA Out Of Memory حتى مع أقل إعدادات. بدل ما السكريبت يقف
    # ويسيب اليوزر من غير نتيجة، بنجرب تنزيل batch/seq تلقائياً
    # خطوة خطوة لحد ما التدريب ينجح أو نوصل لأقل حد ممكن.
    degrade_steps = [
        dict(batch=cfg["batch"], max_seq=cfg["max_seq"], grad_accum=cfg["grad_accum"]),
        dict(batch=1, max_seq=max(cfg["max_seq"] // 2, 128),
             grad_accum=cfg["grad_accum"] * 2),
        dict(batch=1, max_seq=128, grad_accum=cfg["grad_accum"] * 4),
    ]
    # شيل أي تكرار (لو كانت القيم الأصلية أصلاً في أضيق وضع)
    seen = set()
    unique_steps = []
    for s in degrade_steps:
        key = (s["batch"], s["max_seq"], s["grad_accum"])
        if key not in seen:
            seen.add(key)
            unique_steps.append(s)
    degrade_steps = unique_steps

    optim_candidates = ["paged_adamw_32bit", "paged_adamw_8bit", "adamw_torch"]

    adapter_path = os.path.join(args.output_dir, "final_adapter")
    result = None
    training_succeeded = False
    last_error = None
    trainer = None

    import glob

    for degrade_i, dlevel in enumerate(degrade_steps):
        if degrade_i > 0:
            print(f"\n  🔻 تقليل الإعدادات بسبب نفاد الذاكرة (المحاولة {degrade_i+1}): "
                  f"batch={dlevel['batch']}, max_seq={dlevel['max_seq']}, "
                  f"grad_accum={dlevel['grad_accum']}")
            torch.cuda.empty_cache()

        desired = {
            "output_dir": args.output_dir,
            "num_train_epochs": args.epochs,
            "per_device_train_batch_size": dlevel["batch"],
            "per_device_eval_batch_size": 1,
            "eval_accumulation_steps": 1,
            "gradient_accumulation_steps": dlevel["grad_accum"],
            "learning_rate": args.lr,
            "weight_decay": 0.01,
            "warmup_ratio": 0.1,
            "lr_scheduler_type": "cosine",
            "logging_steps": 10,
            "save_strategy": "steps",
            "save_steps": 100,
            "save_total_limit": 1,
            "bf16": False,
            "bf16_full_eval": False,
            "fp16": False,
            "fp16_full_eval": False,
            "tf32": cfg["use_tf32"],
            "max_grad_norm": 0.3,
            "group_by_length": True,
            "report_to": "none",
            "seed": 42,
            "optim": "paged_adamw_32bit",
            "gradient_checkpointing": True,
            "gradient_checkpointing_kwargs": {"use_reentrant": False},
            "packing": False,
            "load_best_model_at_end": False,
            "dataset_text_field": "text",
            "eval_steps": 50,
            "max_length": dlevel["max_seq"],
            "max_seq_length": dlevel["max_seq"],
            "truncation": True,
            "eval_strategy": "steps",
            "evaluation_strategy": "steps",
        }

        training_kwargs = {k: v for k, v in desired.items() if k in sft_params}
        skipped = sorted(set(desired) - set(training_kwargs))
        if skipped and degrade_i == 0:
            print(f"  ℹ️  بارامترات اتجاهلت (مش مدعومة في نسخة trl الحالية): {skipped}")

        trainer_params = inspect.signature(SFTTrainer.__init__).parameters
        if "processing_class" in trainer_params:
            tok_kwarg = {"processing_class": tokenizer}
        else:
            tok_kwarg = {"tokenizer": tokenizer}

        degraded_oom = False
        for attempt_i, optim_name in enumerate(optim_candidates):
            try:
                attempt_kwargs = dict(training_kwargs)
                attempt_kwargs["optim"] = optim_name
                training_args = _build_with_retry(SFTConfig, attempt_kwargs)
                if hasattr(training_args, "bf16"):
                    training_args.bf16 = False
                if hasattr(training_args, "fp16"):
                    training_args.fp16 = False

                trainer = _build_with_retry(
                    SFTTrainer,
                    dict(
                        model=model,
                        args=training_args,
                        train_dataset=train_dataset,
                        eval_dataset=eval_dataset,
                        **tok_kwarg,
                    ),
                )

                torch.cuda.empty_cache()
                print(f"\n  ⏰ بدأ التدريب (optimizer={optim_name}): "
                      f"{datetime.now().strftime('%H:%M:%S')}")
                print("  📊 هتشوف الـ loss كل 10 steps\n")

                result = trainer.train()
                training_succeeded = True
                print(f"\n  ⏰ خلص التدريب : {datetime.now().strftime('%H:%M:%S')}")
                print(f"  📉 Training Loss: {result.training_loss:.4f}")
                print(f"  📈 Total Steps  : {result.global_step}")
                break

            except torch.cuda.OutOfMemoryError as e:
                last_error = e
                print(f"\n  ⚠️  CUDA Out Of Memory مع optimizer={optim_name}: {e}")
                try:
                    del trainer
                except Exception:
                    pass
                import gc
                gc.collect()
                torch.cuda.empty_cache()
                degraded_oom = True
                break  # مفيش فايدة نجرب optimizer تاني — المشكلة في الذاكرة مش الـ optimizer

            except Exception as e:
                last_error = e
                print(f"\n  ⚠️  فشل التدريب بـ optimizer={optim_name}: {e}")
                try:
                    trainer.save_model(adapter_path)
                    tokenizer.save_pretrained(adapter_path)
                    print(f"  🚑 تم حفظ آخر حالة متاحة للموديل (best-effort): {adapter_path}")
                except Exception:
                    pass
                if attempt_i < len(optim_candidates) - 1:
                    print("  🔄 هنجرب optimizer تاني...")
                    torch.cuda.empty_cache()
                    continue

        if training_succeeded:
            break
        if not degraded_oom:
            # فشل لسبب غير متعلق بالذاكرة — تقليل batch/seq مش هيصلّحه
            break
        if degrade_i == len(degrade_steps) - 1:
            print("  ❌ لسه بيطلع Out Of Memory حتى بأقل إعدادات ممكنة.")

    if training_succeeded:
        try:
            trainer.save_model(adapter_path)
            tokenizer.save_pretrained(adapter_path)
            print(f"\n  💾 Adapter محفوظ: {adapter_path}")
        except Exception as e:
            print(f"  ⚠️  فشل حفظ الـ adapter النهائي بالطريقة العادية: {e}")
            training_succeeded = False

    if not training_succeeded:
        ckpts = sorted(
            glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
            key=os.path.getmtime,
        )
        if ckpts:
            adapter_path = ckpts[-1]
            print(f"  ℹ️  هنستخدم آخر checkpoint محفوظ بدالها: {adapter_path}")
        elif os.path.exists(adapter_path):
            print(f"  ℹ️  هنستخدم آخر نسخة best-effort محفوظة: {adapter_path}")
        else:
            print(f"  ❌ مفيش أي adapter أو checkpoint اتحفظ خالص.")
            print(f"     آخر خطأ: {last_error}")
            if isinstance(last_error, torch.cuda.OutOfMemoryError):
                print("\n  💡 الذاكرة المتاحة على الـ GPU ده مش كفاية حتى بأقل")
                print("     إعدادات. الحلول الممكنة:")
                print("     1) جرب GPU أقوى (Colab Pro → A100/L4، أو Runtime →")
                print("        Disconnect and delete runtime ثم وصّل تاني عشان")
                print("        تحرّر أي ذاكرة متسربة من جلسة قديمة)")
                print("     2) قلّل عدد أمثلة الـ Dataset أو قصّرها")
                print("     3) استخدم موديل أصغر (Qwen2.5-3B-Instruct بدل 7B)")
            return None

    try:
        log = {
            "model": args.base_model, "dataset": args.dataset,
            "dataset_size": len(raw_data),
            "train_size": len(train_dataset), "eval_size": len(eval_dataset),
            "epochs": args.epochs, "batch_size": cfg["batch"],
            "grad_accum": cfg["grad_accum"], "learning_rate": args.lr,
            "max_seq_len": cfg["max_seq"], "lora_r": args.lora_r,
            "lora_alpha": args.lora_alpha,
            "training_loss": (result.training_loss if result is not None else None),
            "training_succeeded": training_succeeded,
            "timestamp": datetime.now().isoformat(),
        }
        log_path = os.path.join(args.output_dir, "training_log.json")
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump(log, f, indent=2, ensure_ascii=False)
        print(f"  📝 Log محفوظ   : {log_path}")
    except Exception as e:
        print(f"  ⚠️  فشل حفظ الـ log (مش مشكلة، الـ adapter محفوظ برضه): {e}")

    return adapter_path


# ══════════════════════════════════════════════════════════════════
# 8. TEST MODEL
# ══════════════════════════════════════════════════════════════════
def test_model(args, adapter_path, use_bf16=False):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    sep()
    print("  🧪 الخطوة 8: اختبار النموذج")
    sep()

    compute_dtype = torch.float16

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    from transformers import AutoTokenizer
    base = AutoModelForCausalLM.from_pretrained(
        args.base_model, quantization_config=bnb,
        device_map="auto", trust_remote_code=True,
        token=args.hf_token or None,
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path, trust_remote_code=True
    )
    model.eval()

    def ask(messages, max_new_tokens=400):
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.7, top_p=0.9,
                repetition_penalty=1.1, do_sample=True,
            )
        return tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

    tests = [
        {
            "label": "🇪🇬 عربي — SPD",
            "messages": [
                {"role": "system",
                 "content": "أنت طبيب متخصص في اضطراب المعالجة الحسية وصحة الأطفال."},
                {"role": "user",
                 "content": "ما هو اضطراب المعالجة الحسية؟"},
            ],
        },
        {
            "label": "🌍 English — SPD symptoms",
            "messages": [
                {"role": "system",
                 "content": "You are a medical specialist in Sensory Processing Disorder and child health."},
                {"role": "user",
                 "content": "What are the main symptoms of SPD in children?"},
            ],
        },
        {
            "label": "🇪🇬 عربي — حالة طارئة",
            "messages": [
                {"role": "system",
                 "content": "أنت طبيب أطفال متخصص. تقدم نصائح طبية دقيقة للأهالي."},
                {"role": "user",
                 "content": "ابني عمره 4 سنين وعنده حرارة 39، أعمل إيه؟"},
            ],
        },
    ]

    for t in tests:
        print(f"\n  {'─'*56}")
        print(f"  السؤال [{t['label']}]:")
        print(f"  {t['messages'][-1]['content']}")
        print(f"  {'─'*56}")
        response = ask(t["messages"])
        preview  = response[:500]
        print(f"  الإجابة:\n{preview}")
        if len(response) > 500:
            print("  ... [الباقي محذوف للاختصار]")

    print(f"\n  ✅ الاختبار خلص!")


# ══════════════════════════════════════════════════════════════════
# 9. MERGE ADAPTER
# ══════════════════════════════════════════════════════════════════
def merge_adapter(args, adapter_path):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    sep()
    print("  🔗 الخطوة 9: Merge الـ Adapter مع النموذج")
    print("     ⏳ محتاج VRAM أكبر — الأفضل على A100/L4")
    sep()

    print(f"  🔄 تحميل Base Model بـ float16...")
    base = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=args.hf_token or None,
        low_cpu_mem_usage=True,
    )

    print(f"  🔄 تحميل Adapter من: {adapter_path}")
    peft_model = PeftModel.from_pretrained(base, adapter_path)

    print("  🔄 Merge...")
    merged = peft_model.merge_and_unload()

    os.makedirs(args.merged_dir, exist_ok=True)
    print(f"  💾 حفظ النموذج المدموج في: {args.merged_dir}")
    merged.save_pretrained(args.merged_dir, safe_serialization=True)

    tokenizer = AutoTokenizer.from_pretrained(
        adapter_path, trust_remote_code=True
    )
    tokenizer.save_pretrained(args.merged_dir)

    total_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(args.merged_dir)
        for f in files
    ) / 1024**3

    print(f"  ✅ تم الـ Merge! ({total_gb:.1f} GB)")
    return args.merged_dir


# ══════════════════════════════════════════════════════════════════
# 10. CONVERT TO GGUF
# ══════════════════════════════════════════════════════════════════
def convert_to_gguf(args, merged_path):
    sep()
    print("  📦 الخطوة 10: تحويل لـ GGUF")
    sep()

    os.makedirs(args.gguf_dir, exist_ok=True)
    llama_cpp = "/content/llama.cpp"
    convert_script = os.path.join(llama_cpp, "convert_hf_to_gguf.py")

    if not os.path.exists(llama_cpp):
        print("  📥 تنزيل llama.cpp...")
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/ggerganov/llama.cpp", llama_cpp],
            check=True
        )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", os.path.join(llama_cpp, "requirements.txt")],
        check=True
    )

    f16_path   = os.path.join(args.gguf_dir, "spd-qwen2.5-7b-f16.gguf")
    quant_path = os.path.join(
        args.gguf_dir,
        f"spd-qwen2.5-7b-{args.gguf_quant.lower()}.gguf"
    )

    print(f"  🔄 تحويل لـ GGUF F16...")
    r = subprocess.run(
        [sys.executable, convert_script,
         merged_path, "--outfile", f16_path, "--outtype", "f16"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  ❌ فشل التحويل:\n{r.stderr[-1500:]}")
        _print_manual_gguf(merged_path, f16_path, quant_path, args)
        return None

    f16_gb = os.path.getsize(f16_path) / 1024**3
    print(f"  ✅ GGUF F16 جاهز: {f16_path} ({f16_gb:.1f} GB)")

    if args.gguf_quant == "F16":
        return f16_path

    build_dir = os.path.join(llama_cpp, "build")
    os.makedirs(build_dir, exist_ok=True)
    print(f"  🔨 بناء llama-quantize...")
    # ── علم بناء الـ CUDA الصحيح ──
    # -DLLAMA_CUBLAS=ON كان الاسم القديم وبقى مهجور (deprecated) في
    # نسخ llama.cpp الحديثة، اللي بقت بتستخدم -DGGML_CUDA=ON. لو
    # استخدمنا الاسم القديم بس، الـ build هيتم لكن من غير دعم CUDA
    # فعلي (أو هيفشل تماماً حسب النسخة)، فبنجرب الاسم الجديد الأول
    # وبعدين القديم كـ fallback.
    cmake_configured = subprocess.run(
        ["cmake", "..", "-DGGML_CUDA=ON"],
        cwd=build_dir, capture_output=True, text=True
    )
    if cmake_configured.returncode != 0:
        print("  ℹ️  -DGGML_CUDA=ON فشل، هنجرب الاسم القديم -DLLAMA_CUBLAS=ON")
        subprocess.run(["cmake", "..", "-DLLAMA_CUBLAS=ON"],
                       cwd=build_dir, capture_output=True)
    subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j4"],
                   cwd=build_dir, capture_output=True)

    quantize_bin = None
    for candidate in [
        os.path.join(build_dir, "bin", "llama-quantize"),
        os.path.join(build_dir, "bin", "llama-quantize.exe"),
        shutil.which("llama-quantize"),
    ]:
        if candidate and os.path.exists(candidate):
            quantize_bin = candidate
            break

    if not quantize_bin:
        print(f"  ⚠️  llama-quantize مش موجود — F16 متاح: {f16_path}")
        _print_manual_gguf(merged_path, f16_path, quant_path, args)
        return f16_path

    print(f"  🔄 تشفير بـ {args.gguf_quant}...")
    r = subprocess.run(
        [quantize_bin, f16_path, quant_path, args.gguf_quant],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        q_gb = os.path.getsize(quant_path) / 1024**3
        os.remove(f16_path)
        print(f"  ✅ GGUF {args.gguf_quant} جاهز: {quant_path} ({q_gb:.1f} GB)")
        print(f"  🗑️  حذف F16 المؤقت")
        return quant_path
    else:
        print(f"  ⚠️  فشل التشفير — F16 لا زال متاح: {f16_path}")
        return f16_path


def _print_manual_gguf(merged_path, f16_path, quant_path, args):
    print("\n  💡 للتحويل اليدوي:")
    print(f"     python llama.cpp/convert_hf_to_gguf.py {merged_path} \\")
    print(f"       --outfile {f16_path} --outtype f16")
    print(f"     llama-quantize {f16_path} {quant_path} {args.gguf_quant}")


# ══════════════════════════════════════════════════════════════════
# 11. UPLOAD TO HUGGINGFACE HUB (optional)
# ══════════════════════════════════════════════════════════════════
def upload_to_hf(args, adapter_path):
    sep()
    print("  ☁️  الخطوة 11: رفع على HuggingFace Hub")
    sep()

    if not args.hf_token:
        print("  ❌ محتاج --hf_token")
        return
    if not args.hf_repo:
        print("  ❌ محتاج --hf_repo  مثال: username/spd-qwen2.5-7b")
        return

    from huggingface_hub import HfApi
    api = HfApi(token=args.hf_token)

    try:
        api.create_repo(repo_id=args.hf_repo, private=True, exist_ok=True)
    except Exception as e:
        print(f"  ⚠️  {e}")

    print(f"  🔄 رفع الـ Adapter على: {args.hf_repo}")
    api.upload_folder(
        folder_path=adapter_path,
        repo_id=args.hf_repo,
        repo_type="model",
    )
    print(f"  ✅ تم الرفع!")
    print(f"     🔗 https://huggingface.co/{args.hf_repo}")


# ══════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════
def sep(char="═", width=60):
    print(char * width)


def _build_with_retry(cls, kwargs, max_attempts=25):
    import re
    kwargs = dict(kwargs)
    for _ in range(max_attempts):
        try:
            return cls(**kwargs)
        except TypeError as e:
            msg = str(e)
            match = re.search(r"unexpected keyword argument '([^']+)'", msg)
            if not match:
                raise
            bad_key = match.group(1)
            if bad_key not in kwargs:
                raise
            print(f"  ⚠️  {cls.__name__}: البارامتر '{bad_key}' غير مدعوم "
                  f"في النسخة المثبتة — هيتم تجاهله")
            del kwargs[bad_key]
    raise RuntimeError(
        f"❌ فشل بناء {cls.__name__} بعد {max_attempts} محاولة لإزالة بارامترات"
    )


def login_hf(token):
    if token:
        from huggingface_hub import login
        login(token=token)
        print("  ✅ تم تسجيل الدخول في HuggingFace")
    else:
        print("  ⚠️  لا يوجد HF Token — لو النموذج private هتحتاجه")
        print("     احصل عليه من: https://huggingface.co/settings/tokens")
        print("     وضعه في: --hf_token hf_xxx")


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════
def main():
    args = parse_args()

    sep()
    print("  🧠 SPD & Child Health — Qwen2.5-7B Fine-Tuning")
    print(f"  📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  🤖 النموذج  : {args.base_model}")
    print(f"  📂 Dataset  : {args.dataset}")
    print(f"  💾 Output   : {args.output_dir}")
    sep()

    for d in [args.output_dir, args.merged_dir, args.gguf_dir]:
        os.makedirs(d, exist_ok=True)

    install_dependencies()
    login_hf(args.hf_token)
    cfg = check_gpu_and_set_config()

    if args.lora_r is None:
        args.lora_r = cfg["lora_r"]
    if args.lora_alpha is None:
        args.lora_alpha = args.lora_r * 2

    adapter_path = os.path.join(args.output_dir, "final_adapter")

    if not args.skip_train and not args.merge_only:

        raw_data = load_dataset(args.dataset)
        tokenizer = load_tokenizer(args.base_model, args.hf_token)
        train_ds, eval_ds = format_dataset(
            raw_data, tokenizer, cfg["max_seq"]
        )
        model = load_model_with_lora(
            args.base_model, args.hf_token,
            args.lora_r, args.lora_alpha, args.lora_dropout,
            use_bf16=cfg["use_bf16"],
        )

        adapter_path = train(
            args, cfg, model, tokenizer, train_ds, eval_ds, raw_data
        )

        import torch, gc
        del model
        gc.collect()
        torch.cuda.empty_cache()

        if adapter_path is None:
            sep()
            print("  ❌ مفيش adapter اتحفظ — التدريب فشل تماماً من غير "
                  "أي نتيجة قابلة للاستخدام.")
            print("     راجع رسائل الخطأ اللي فوق.")
            sep()
            return

        try:
            test_model(args, adapter_path, use_bf16=cfg["use_bf16"])
        except Exception as e:
            print(f"  ⚠️  فشل اختبار الموديل (الـ adapter لسه محفوظ "
                  f"وسليم): {e}")

        if args.upload_hf:
            try:
                upload_to_hf(args, adapter_path)
            except Exception as e:
                print(f"  ⚠️  فشل الرفع على HuggingFace (الـ adapter "
                      f"لسه محفوظ محلياً): {e}")

    merged_path = None
    if args.merge_only or (
        os.path.exists(adapter_path) and not args.skip_train
    ):
        import torch, gc
        gc.collect()
        torch.cuda.empty_cache()

        try:
            merged_path = merge_adapter(args, adapter_path)
        except Exception as e:
            print(f"  ⚠️  فشل الـ Merge: {e}")
            print(f"     ولكن الـ LoRA Adapter لسه محفوظ وسليم في: {adapter_path}")
            print(f"     تقدر تعمل merge بعدين بـ:")
            print(f"     python {SCRIPT_NAME} --merge_only")

        if merged_path and args.convert_gguf:
            try:
                convert_to_gguf(args, merged_path)
            except Exception as e:
                print(f"  ⚠️  فشل تحويل GGUF: {e}")
                print(f"     ولكن الموديل المدموج لسه موجود في: {merged_path}")

    sep()
    print("  🎉 خلصنا!")
    sep()
    print(f"\n  📁 المسارات:")
    print(f"     LoRA Adapter : {adapter_path}")
    if merged_path and os.path.exists(args.merged_dir) and os.listdir(args.merged_dir):
        print(f"     Merged Model : {args.merged_dir}")
    if os.path.exists(args.gguf_dir) and os.listdir(args.gguf_dir):
        print(f"     GGUF Model   : {args.gguf_dir}")
    print(f"\n  🚀 الخطوات الجاية:")
    print(f"     Merge : python {SCRIPT_NAME} --merge_only")
    print(f"     GGUF  : python {SCRIPT_NAME} --merge_only --convert_gguf")
    print(f"     رفع HF: python {SCRIPT_NAME} --skip_train --upload_hf \\")
    print(f"               --hf_token hf_xxx --hf_repo username/repo-name")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        # ── مش خطأ في الكود — المستخدم ضغط Stop أو Ctrl+C ──
        # الـ pip install كانت بتبان فريزة (capture_output=True صامتة)
        # فالمستخدم بيضغط Stop ظناً إنها واقفة. دلوقتي الـ output
        # بيطلع مباشرةً ومحتاج بس ينتظر.
        sep()
        print("\n  ⏹️  توقف يدوي (Ctrl+C أو ضغطت Stop في Colab)")
        print("     لو التدريب كان بدأ، هتلاقي آخر checkpoint محفوظ في:")
        _args = parse_args()
        print(f"     {_args.output_dir}/")
        print("\n  💡 الخطوات الجاية:")
        print(f"     عشان تكمل: شغّل السكريبت تاني بنفس الأوامر")
        print(f"     عشان تعمل merge فقط: python {SCRIPT_NAME} --merge_only")
        sep()
    except Exception as e:
        sep()
        print(f"  ❌ حصل خطأ مش متوقّع أوقف السكريبت: {e}")
        print("     لو كان فيه adapter اتحفظ قبل الخطأ ده، هتلاقيه في:")
        _args = parse_args()
        print(f"     {_args.output_dir}/final_adapter "
              f"أو جوه مجلد checkpoint-* بنفس المسار.")
        raise

════════════════════════════════════════════════════════════
  🧠 SPD & Child Health — Qwen2.5-7B Fine-Tuning
  📅 2026-06-22 22:24:57
  🤖 النموذج  : Qwen/Qwen2.5-7B-Instruct
  📂 Dataset  : /content/SPD_finetune_dataset.json
  💾 Output   : /content/spd_finetuned_output
════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════
  📦 الخطوة 1: تثبيت المكتبات
════════════════════════════════════════════════════════════
  ✅ PyTorch موجود بالفعل: 2.11.0+cu128
  ⏳ تثبيت مكتبات التدريب (transformers, peft, trl, ...) (ممكن ياخد دقيقتين-5، استنى...)
  ✅ كل المكتبات اتثبتت!
  ⚠️  لا يوجد HF Token — لو النموذج private هتحتاجه
     احصل عليه من: https://huggingface.co/settings/tokens
     وضعه في: --hf_token hf_xxx
════════════════════════════════════════════════════════════
  🖥️  الخطوة 2: فحص GPU وضبط الإعدادات
════════════════════════════════════════════════════════════
  🖥️  GPU      : Tesla T4
  💾 VRAM كلي : 14.6 GB
  ✅ VRAM حر  : 14

Saving SPD_finetune_dataset.json to SPD_finetune_dataset.json
  ✅ تم رفع الملف: /content/SPD_finetune_dataset.json
  ✅ إجمالي الأمثلة : 126
  ✅ أمثلة صالحة   : 126/126

  📝 مثال من الـ Dataset:
     [SYSTEM   ]: أنت طبيب وأخصائي متخصص في اضطراب المعالجة الحسية (Sensory Processing Disorder - SPD). تقدم معلومات ط...
     [USER     ]: ما هو اضطراب المعالجة الحسية؟...
     [ASSISTANT]: اضطراب المعالجة الحسية (SPD) هو خلل عصبي يؤثر على طريقة استقبال الدماغ للمعلومات الحسية ومعالجتها وا...
════════════════════════════════════════════════════════════
  🔤 الخطوة 4: تحميل Tokenizer
     النموذج: Qwen/Qwen2.5-7B-Instruct
════════════════════════════════════════════════════════════


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...


  ────────────────────────────────────────────────────────
  السؤال [🇪🇬 عربي — SPD]:
  ما هو اضطراب المعالجة الحسية؟
  ────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  الإجابة:
اضطراب المعالجة الحسية (Sensory Processing Disorder - SPD) هو صعوبة الدماغ في معالجة المعلومات الحسية الواردة من الجسم والعالم المحيط به.

**ما هو التحدي الأساسي:**
- الدماغ لا يستطيع فهم أو تنظيم الإشارات الحسية بشكل طبيعي
- الطفل قد يشعر بالخوف والقلق بسبب الإحساس بـ "ال mundane" كأنه "الأصوات العالية" أو "اللمسة الخفيفة"

**أنواع المعالجة الحسية:**
1. **الحس السمعي:** أصوات مزعجة (السيارات، الأصوات العامة)
2. **الحس البصري:** إضاءة ساطعة، حركة غير متوقعة
3. **الحس الجذعي:** اللمس، القلبان (مث
  ... [الباقي محذوف للاختصار]

  ────────────────────────────────────────────────────────
  السؤال [🌍 English — SPD symptoms]:
  What are the main symptoms of SPD in children?
  ────────────────────────────────────────────────────────
  الإجابة:
Sensory Processing Disorder (SPD) affects how a child perceives, interprets, and responds to sensory information from their environment. Here are the primary symptoms:

**1. Tactile Defensiveness:**
- Refusing to touch certain textures or mat

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  🔄 تحميل Adapter من: /content/spd_finetuned_output/final_adapter
  ⚠️  فشل الـ Merge: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: base_model.model.model.layers.5, base_model.model.model.layers.6, base_model.model.model.layers.7, base_model.model.model.layers.8, base_model.model.model.layers.9, base_model.model.model.layers.10, base_model.model.model.layers.11, base_model.model.model.layers.12, base_model.model.model.layers.13, base_model.model.model.layers.14, base_model.model.model.layers.15, base_model.model.model.layers.16, base_model.model.model.layers.17, base_model.model.model.layers.18, base_model.model.model.layers.19, base_model.model.model.layers.20, base_model.model.model.layers.21, base_model.model.model.layers.22, base_model.model.model.layers.23, base_model.model.model.layers.24, base_model.model.model.layers.25, base_model.model.model.layers.26, base_model.model.model.layers.27, base_mode